In [17]:
"""
Módulo 1: Treinamento de modelos CNN
Saída: Modelos CNN treinados salvos em SAVE_DIR
"""
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
import timm
import pickle
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

from config import *
EPOCHS = 60


In [18]:
# Dataset
class LeukemiaDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Carregar imagens ALL (classe 1)
        all_dir = os.path.join(data_dir, 'all')
        if os.path.exists(all_dir):
            for img_name in os.listdir(all_dir):
                if img_name.endswith(('.jpg', '.png', '.bmp')):
                    self.images.append(os.path.join(all_dir, img_name))
                    self.labels.append(1)
        
        # Carregar imagens HEM (classe 0)
        hem_dir = os.path.join(data_dir, 'hem')
        if os.path.exists(hem_dir):
            for img_name in os.listdir(hem_dir):
                if img_name.endswith(('.jpg', '.png', '.bmp')):
                    self.images.append(os.path.join(hem_dir, img_name))
                    self.labels.append(0)
        
        print(f"Carregado: {len(self.images)} imagens - ALL: {self.labels.count(1)}, HEM: {self.labels.count(0)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Erro ao carregar {img_path}: {e}")
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color='black')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [19]:
# Transformações

def get_transforms():
    train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),  # Menos rotação
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Menos variação
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

In [20]:
# Modelo com Attention
class CNNModel(nn.Module):
    def __init__(self, model_name, num_classes=2):
        super().__init__()
        # Criar backbone com o model_name passado como parâmetro
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,  # Não baixar pesos pré-treinados (vamos carregar do checkpoint)
            num_classes=0,
            global_pool='avg',
            drop_rate=0.3,
            drop_path_rate=0.2
        )
        
        num_features = self.backbone.num_features
        
        # Usar exatamente a mesma arquitetura do classificador do treinamento
        self.classifier = nn.Sequential(
            nn.LayerNorm(num_features),
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

In [21]:
def mixup_data(x, y, alpha=0.2):
    """Aplica mixup aos dados"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Calcula loss com mixup"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_epoch(model, loader, criterion, optimizer, scaler, device, use_mixup=True):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    for inputs, targets in tqdm(loader, desc='Training'):
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        # Aplicar Mixup com probabilidade
        use_mixup_batch = use_mixup and np.random.random() > 0.5
        
        if use_mixup_batch:
            inputs_mixed, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha=0.3)
            
            if scaler is not None:
                with autocast():
                    outputs = model(inputs_mixed)
                    loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            else:
                outputs = model(inputs_mixed)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            if scaler is not None:
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
        
        # Backward pass
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        
        # Para métricas, usar targets originais (não mixados)
        _, preds = torch.max(outputs, 1)
        if use_mixup_batch:
            # Para mixup, usar o target dominante para métricas
            targets_metrics = targets_a if lam > 0.5 else targets_b
        else:
            targets_metrics = targets
            
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets_metrics.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='weighted')
    
    return avg_loss, acc, f1

In [22]:
# Função de validação
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc='Validation'):
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='weighted')
    
    return avg_loss, acc, f1

In [23]:
def train_single_model(model_name):
    """Treina um único modelo CNN"""
    print(f"\n{'='*50}")
    print(f"Treinando: {model_name}")
    print('='*50)
    
    # Preparar datasets
    train_transform, val_transform = get_transforms()
    train_dataset = LeukemiaDataset(TRAIN_DIR, train_transform)
    val_dataset = LeukemiaDataset(VAL_DIR, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    # Criar modelo
    model = CNNModel(model_name, NUM_CLASSES).to(DEVICE)
    
    # Configurar treinamento
    # Calcular pesos das classes baseado no desbalanceamento
    def calculate_class_weights(train_dataset):
        labels = np.array(train_dataset.labels)
        class_counts = np.bincount(labels)
        total = len(labels)
        weights = total / (len(class_counts) * class_counts)
        return torch.FloatTensor(weights)

    # Configurar treinamento
    class_weights = calculate_class_weights(train_dataset).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # Optimizer com diferentes learning rates
    param_groups = [
        {'params': model.backbone.parameters(), 'lr': LEARNING_RATE * 0.5},  # LR menor para backbone
        {'params': model.classifier.parameters(), 'lr': LEARNING_RATE}
    ]
    optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY * 2)  # Mais weight decay
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

    # Early stopping mais rigoroso
    class EarlyStopping:
        def __init__(self, patience=10, min_delta=0.001):
            self.patience = patience
            self.min_delta = min_delta
            self.counter = 0
            self.best_score = None
            self.early_stop = False
            
        def __call__(self, val_score):
            if self.best_score is None:
                self.best_score = val_score
            elif val_score < self.best_score + self.min_delta:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_score = val_score
                self.counter = 0
            return self.early_stop



    scaler = GradScaler() if DEVICE.type == 'cuda' else None
    
    # Treinar
    best_f1 = 0
    best_gap = float('inf')  # Menor diferença entre train e val
    early_stopping = EarlyStopping(patience=30, min_delta=0.002)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        # Treinar
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
        
        # Validar
        val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, DEVICE)
        
        # Atualizar scheduler
        scheduler.step(val_loss)
        
        # Registrar métricas
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        
        print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        

        gap = abs(train_f1 - val_f1)

        # Salvar melhor modelo
        if val_f1 > 0.85 and gap < best_gap:  # Só salva se F1 > 0.85 e gap menor
            best_gap = gap
            best_f1 = val_f1
            model_path = os.path.join(SAVE_DIR, f'{model_name.replace("/", "_")}_best.pt')

            # Certificar-se de que estamos salvando corretamente
            checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'best_f1': best_f1,
            'val_f1': val_f1,
            'train_f1': train_f1,
            'gap': gap,
            'val_loss': val_loss
            }

            # Verificações antes de salvar
            assert isinstance(checkpoint, dict), "Checkpoint deve ser dict"
            assert 'model_state_dict' in checkpoint, "model_state_dict ausente"
            assert isinstance(checkpoint['model_state_dict'], dict), "state_dict deve ser dict"

            # Criar diretório se não existir
            os.makedirs(SAVE_DIR, exist_ok=True)
 
            torch.save(checkpoint, model_path)
            print(f"✅ Modelo salvo - F1: {best_f1:.4f}, Gap: {gap:.4f}")
        
        # Early stopping
        #if early_stopping(val_f1):
        #    print("Early stopping triggered!")
        #    break
    
    # Salvar histórico
    history_path = os.path.join(SAVE_DIR, f'{model_name.replace("/", "_")}_history.pkl')
    with open(history_path, 'wb') as f:
        pickle.dump(history, f)
    
    return best_f1

In [24]:
def main():
    print("Iniciando treinamento de modelos CNN...")
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    results = {}
    for model_name in CNN_MODELS:
        try:
            best_f1 = train_single_model(model_name)
            results[model_name] = best_f1
            
            # Limpar memória
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"Erro ao treinar {model_name}: {e}")
            results[model_name] = 0
    
    print("\n" + "="*50)
    print("RESULTADOS FINAIS:")
    for name, f1 in results.items():
        print(f"{name}: F1-Score = {f1:.4f}")
    
    # Salvar resumo
    with open(os.path.join(SAVE_DIR, 'cnn_training_results.pkl'), 'wb') as f:
        pickle.dump(results, f)

if __name__ == "__main__":
    main()

Iniciando treinamento de modelos CNN...

Treinando: tf_efficientnetv2_b3
Carregado: 11102 imagens - ALL: 5335, HEM: 5767
Carregado: 1172 imagens - ALL: 571, HEM: 601

Epoch 1/60


Validation: 100%|██████████| 147/147 [00:10<00:00, 13.66it/s]


Train Loss: 0.5123 | Train F1: 0.8195
Val Loss: 0.3007 | Val F1: 0.9454
✅ Modelo salvo - F1: 0.9454, Gap: 0.1258

Epoch 2/60


Validation: 100%|██████████| 147/147 [00:10<00:00, 14.56it/s]


Train Loss: 0.4529 | Train F1: 0.8691
Val Loss: 0.2932 | Val F1: 0.9419
✅ Modelo salvo - F1: 0.9419, Gap: 0.0728

Epoch 3/60


...

Train Loss: 0.2853 | Train F1: 0.9754
Val Loss: 0.3047 | Val F1: 0.9470

Epoch 59/60


Validation: 100%|██████████| 147/147 [00:10<00:00, 14.33it/s]


Train Loss: 0.2786 | Train F1: 0.9743
Val Loss: 0.3429 | Val F1: 0.9270

Epoch 60/60


Validation: 100%|██████████| 147/147 [00:09<00:00, 15.79it/s]

Train Loss: 0.2814 | Train F1: 0.9726
Val Loss: 0.2651 | Val F1: 0.9633

RESULTADOS FINAIS:
tf_efficientnetv2_b3: F1-Score = 0.9744
